# Create Work Fulltext

## Overview

Incrementally maintains the precomputed fulltext used by `CreateWorksEnriched`
(oxjob #628). Replaces the old Enriched cell that re-scanned and regex-cleaned
all of `openalex.pdf.pdf_combined` every day: here, only PDF rows ingested since
the last run are cleaned, and only works whose matched fulltext changed are
rewritten.

## Data Flow

```
openalex.pdf.pdf_combined (ingested_at watermark)
    |
    v  daily delta only: extract keys, pick longest, clean
openalex.works.pdf_fulltext_keys      (key_type/key -> cleaned fulltext)
    |
    v  match against openalex_works_base (doi > pmh), slim keys first
openalex.works.work_fulltext          (work_id -> fulltext)
    |
    v
CreateWorksEnriched (lean MERGE, positioned last so the other merges
                     run on the slim table)
```

## Semantics (parity with the old Enriched fulltext cell)

- Keys: DOI (`https://doi.org/` + lowercased) when the PDF has one; PMH id only
  for PDFs with no DOI.
- Per key, the longest **raw** fulltext wins. Replacement across runs requires a
  strictly longer raw fulltext; ties keep the existing value. (The old window
  ordered by `LENGTH DESC` with no tie-break and could flap between equal-length
  variants — this is deliberately deterministic.)
- Cleaning: 200K cap -> strip HTML tags -> collapse whitespace -> trim.
- Per work, DOI match beats PMH match. Among several PMH matches, the longest
  raw fulltext wins (old behavior was nondeterministic here).
- Merge-only: a work that loses its PDF row keeps its fulltext, keys are never
  deleted — identical to the old MERGE, which never NULLed fulltext out.

## First run

Self-seeding: with `pdf_fulltext_keys` empty, the watermark is NULL and the
delta is the full corpus (~62M fulltexts get cleaned once). Run this notebook
manually on a large warehouse before wiring it into end2end so the scheduled
runs only ever see the daily delta.


### State tables (no-ops after first run)

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys') (
  key_type STRING NOT NULL,  -- 'doi' | 'pmh'
  key STRING NOT NULL,       -- doi: 'https://doi.org/<lowercased doi>'; pmh: raw pmh id
  fulltext STRING,
  fulltext_length BIGINT,
  raw_length BIGINT,         -- raw pdf_combined fulltext length; replacement compares this
  src_ingested_at TIMESTAMP,
  updated_at TIMESTAMP
)
CLUSTER BY (key_type, key);

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.work_fulltext') (
  work_id BIGINT NOT NULL,
  fulltext STRING,
  match_type STRING,  -- 'doi' | 'pmh'
  key STRING,
  raw_length BIGINT,
  updated_at TIMESTAMP
)
CLUSTER BY (work_id);

### Incremental key-level build: clean only PDFs ingested since the last run

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys_delta') AS
WITH watermark AS (
  -- 1h lookback: idempotent overlap guards ingested_at commit races.
  -- Empty table (first run) => NULL => full-corpus seed.
  SELECT COALESCE(MAX(src_ingested_at) - INTERVAL 1 HOUR, TIMESTAMP '1900-01-01') AS wm
  FROM identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys')
),
delta_pdfs AS (
  SELECT p.ids, p.fulltext, p.ingested_at
  FROM openalex.pdf.pdf_combined p, watermark w
  WHERE p.ingested_at > w.wm
    AND p.fulltext IS NOT NULL
    AND TRIM(p.fulltext) != ''
),
keyed AS (
  SELECT
    'doi' AS key_type,
    CONCAT('https://doi.org/', LOWER(FILTER(ids, x -> x.namespace = 'doi')[0].id)) AS key,
    fulltext,
    ingested_at
  FROM delta_pdfs
  WHERE SIZE(FILTER(ids, x -> x.namespace = 'doi')) > 0

  UNION ALL

  SELECT
    'pmh' AS key_type,
    FILTER(ids, x -> x.namespace = 'pmh')[0].id AS key,
    fulltext,
    ingested_at
  FROM delta_pdfs
  WHERE SIZE(FILTER(ids, x -> x.namespace = 'pmh')) > 0
    AND SIZE(FILTER(ids, x -> x.namespace = 'doi')) = 0
),
winners AS (
  SELECT
    key_type,
    key,
    fulltext,
    LENGTH(fulltext) AS raw_length,
    ROW_NUMBER() OVER (
      PARTITION BY key_type, key
      ORDER BY LENGTH(fulltext) DESC, XXHASH64(fulltext) ASC
    ) AS rn,
    MAX(ingested_at) OVER (PARTITION BY key_type, key) AS max_ingested_at
  FROM keyed
  WHERE key IS NOT NULL
),
cleaned AS (
  SELECT
    key_type,
    key,
    raw_length,
    max_ingested_at,
    TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(SUBSTRING(fulltext, 1, 200000), '<[^>]+>', ' '),
          '\\s+', ' '
        ),
        '(^\\s+|\\s+$)', ''
      )
    ) AS fulltext
  FROM winners
  WHERE rn = 1
)
SELECT
  key_type,
  key,
  fulltext,
  LENGTH(fulltext) AS fulltext_length,
  raw_length,
  max_ingested_at AS src_ingested_at
FROM cleaned
WHERE fulltext IS NOT NULL AND LENGTH(fulltext) > 0;

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys') AS t
USING identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys_delta') AS s
ON t.key_type = s.key_type AND t.key = s.key
WHEN MATCHED THEN UPDATE SET
  t.fulltext        = CASE WHEN s.raw_length > t.raw_length THEN s.fulltext        ELSE t.fulltext        END,
  t.fulltext_length = CASE WHEN s.raw_length > t.raw_length THEN s.fulltext_length ELSE t.fulltext_length END,
  t.raw_length      = CASE WHEN s.raw_length > t.raw_length THEN s.raw_length      ELSE t.raw_length      END,
  t.updated_at      = CASE WHEN s.raw_length > t.raw_length THEN CURRENT_TIMESTAMP() ELSE t.updated_at    END,
  -- always advance, or skipped shorter duplicates would be rescanned forever
  t.src_ingested_at = GREATEST(t.src_ingested_at, s.src_ingested_at)
WHEN NOT MATCHED THEN INSERT
  (key_type, key, fulltext, fulltext_length, raw_length, src_ingested_at, updated_at)
  VALUES (s.key_type, s.key, s.fulltext, s.fulltext_length, s.raw_length, s.src_ingested_at, CURRENT_TIMESTAMP());

### Work-level matching against `openalex_works_base` (slim keys first, bytes only for changed rows)

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.work_fulltext_matches') AS
WITH doi_matches AS (
  SELECT w.work_id, k.key_type, k.key, k.raw_length, 1 AS priority
  FROM (
    SELECT id AS work_id, LOWER(doi) AS doi_lower
    FROM identifier('openalex' || :env_suffix || '.works.openalex_works_base')
    WHERE doi IS NOT NULL
  ) w
  JOIN (
    SELECT key_type, key, raw_length
    FROM identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys')
    WHERE key_type = 'doi'
  ) k ON k.key = w.doi_lower
),
pmh_pairs AS (
  SELECT DISTINCT work_id, loc.pmh_id
  FROM (
    SELECT id AS work_id, EXPLODE(locations) AS loc
    FROM identifier('openalex' || :env_suffix || '.works.openalex_works_base')
  )
  WHERE loc.pmh_id IS NOT NULL
),
pmh_matches AS (
  SELECT p.work_id, k.key_type, k.key, k.raw_length, 2 AS priority
  FROM pmh_pairs p
  JOIN (
    SELECT key_type, key, raw_length
    FROM identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys')
    WHERE key_type = 'pmh'
  ) k ON k.key = p.pmh_id
),
best AS (
  SELECT
    work_id, key_type, key, raw_length,
    ROW_NUMBER() OVER (
      PARTITION BY work_id
      ORDER BY priority ASC, raw_length DESC, key ASC
    ) AS rn
  FROM (
    SELECT * FROM doi_matches
    UNION ALL
    SELECT * FROM pmh_matches
  )
)
SELECT work_id, key_type, key, raw_length
FROM best
WHERE rn = 1;

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.work_fulltext_delta') AS
WITH changed AS (
  -- (match_type, key, raw_length) identifies content: a key's fulltext only
  -- changes when its raw_length strictly grows
  SELECT m.work_id, m.key_type, m.key, m.raw_length
  FROM identifier('openalex' || :env_suffix || '.works.work_fulltext_matches') m
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.work_fulltext') t
    ON t.work_id = m.work_id
  WHERE t.work_id IS NULL
     OR t.match_type != m.key_type
     OR t.key != m.key
     OR t.raw_length != m.raw_length
)
SELECT c.work_id, k.fulltext, c.key_type AS match_type, c.key, c.raw_length
FROM changed c
JOIN identifier('openalex' || :env_suffix || '.works.pdf_fulltext_keys') k
  ON k.key_type = c.key_type AND k.key = c.key;

In [0]:
-- merge-only: no delete path, matching the openalex_works fulltext MERGE
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_fulltext') AS t
USING identifier('openalex' || :env_suffix || '.works.work_fulltext_delta') AS s
ON t.work_id = s.work_id
WHEN MATCHED THEN UPDATE SET
  t.fulltext   = s.fulltext,
  t.match_type = s.match_type,
  t.key        = s.key,
  t.raw_length = s.raw_length,
  t.updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (work_id, fulltext, match_type, key, raw_length, updated_at)
  VALUES (s.work_id, s.fulltext, s.match_type, s.key, s.raw_length, CURRENT_TIMESTAMP());